# 02 — Cointegration and Spread Construction

**Objective:** test whether KO and PEP have a stable long-run relationship that can be represented by a mean-reverting spread.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm

from statsmodels.tsa.stattools import adfuller, coint

TICKERS = ["KO", "PEP"]

prices = yf.download(
    TICKERS,
    start="2018-01-01",
    end="2026-01-01",
    auto_adjust=True,
    progress=False
)["Close"].dropna()

## Stationarity

I use the Augmented Dickey-Fuller test to check whether the individual price series are stationary.

In [ ]:
def adf_pvalue(series):
    return adfuller(series.dropna())[1]

for ticker in TICKERS:
    print(f"{ticker} ADF p-value: {adf_pvalue(prices[ticker]):.4f}")

## Hedge ratio and spread

I regress KO on PEP and use the fitted relationship to construct the spread:

\[
spread_t = KO_t - (\alpha + \beta PEP_t)
\]

In [ ]:
y = prices["KO"]
x = prices["PEP"]

model = sm.OLS(y, sm.add_constant(x)).fit()

alpha = model.params["const"]
beta = model.params["PEP"]

spread = y - (alpha + beta * x)

print(f"Alpha: {alpha:.4f}")
print(f"Beta:  {beta:.4f}")
print(f"Spread ADF p-value: {adf_pvalue(spread):.4f}")

In [ ]:
spread.plot(figsize=(10, 4))
plt.axhline(spread.mean(), linestyle="--")
plt.title("KO/PEP Estimated Spread")
plt.xlabel("Date")
plt.ylabel("Spread")
plt.grid(alpha=0.3)
plt.show()

## Cointegration test

A low p-value provides evidence that the two non-stationary price series share a stable long-run relationship.

In [ ]:
stat, pvalue, _ = coint(y, x)

print(f"Cointegration statistic: {stat:.4f}")
print(f"Cointegration p-value:   {pvalue:.4f}")